In [1]:
from langchain_chroma import Chroma
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


C:\Users\goran\AppData\Local\Temp\ipykernel_17396\2282627514.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
# Initialize embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)


In [3]:
# Load and split documents
documents_path = Path(r"c:\Artificial Intelligence\rag_tutorial\documents")

loader = DirectoryLoader(
    str(documents_path),
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
documents = loader.load()


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splits = text_splitter.split_documents(documents)


In [5]:
# Create persistent Chroma database (langchain-chroma writes to persist_directory automatically)
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="my_collection"
)


In [6]:
# Load existing database
loaded_vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,
    collection_name="my_collection"
)

In [7]:
# Advanced search with scores
results_with_scores = loaded_vectorstore.similarity_search_with_score(
    "How to build RAG applications?",
    k=5
)

In [8]:
# Advanced search with scores
results_with_scores_2 = vectorstore.similarity_search_with_score(
    "How to build RAG applications?",
    k=5
)

In [9]:
for doc, score in results_with_scores:
    print(f"Score: {score:.3f}")
    print(f"Content: {doc.page_content[:100]}...")
    print(f"Metadata: {doc.metadata}\n")

Score: 0.688
Content: The Transformer architecture, introduced in 2017, has largely replaced RNNs for
language processing ...
Metadata: {'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\neural-networks.txt'}

Score: 0.694
Content: The **attention mechanism** and subsequent **Transformer** architecture resolved
these limitations t...
Metadata: {'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\ai-intro.txt'}

Score: 0.696
Content: The trajectory toward **Artificial General Intelligence** remains the field's
most contested and con...
Metadata: {'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\ai-intro.txt'}

Score: 0.700
Content: Transfer learning has become a standard practice that dramatically reduces the
data and compute requ...
Metadata: {'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\neural-networks.txt'}

Score: 0.708
Content: Within AI research, a fundamental distinction separates **Strong AI** (or
Artificial General

In [10]:
for doc, score in results_with_scores_2:
    print(f"Score: {score:.3f}")
    print(f"Content: {doc.page_content[:100]}...")
    print(f"Metadata: {doc.metadata}\n")

Score: 0.688
Content: The Transformer architecture, introduced in 2017, has largely replaced RNNs for
language processing ...
Metadata: {'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\neural-networks.txt'}

Score: 0.694
Content: The **attention mechanism** and subsequent **Transformer** architecture resolved
these limitations t...
Metadata: {'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\ai-intro.txt'}

Score: 0.696
Content: The trajectory toward **Artificial General Intelligence** remains the field's
most contested and con...
Metadata: {'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\ai-intro.txt'}

Score: 0.700
Content: Transfer learning has become a standard practice that dramatically reduces the
data and compute requ...
Metadata: {'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\neural-networks.txt'}

Score: 0.708
Content: Within AI research, a fundamental distinction separates **Strong AI** (or
Artificial General

Hybrid Search Implementation

In [11]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

In [12]:
# Create vector store for semantic search
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,
    collection_name="my_collection"
)

In [13]:
# semantic retriever
vector_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

In [14]:
# Create BM25 retriever for keyword search
data = vectorstore.get()  # dict with "documents" and "metadatas"
bm25_docs = [
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(data["documents"], data["metadatas"])
]
bm25_retriever = BM25Retriever.from_documents(bm25_docs) # keyword search
bm25_retriever.k = 5

In [15]:
# Combine retrievers with weights
ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.7, 0.3]  # 70% semantic, 30% keyword
)

In [16]:
# Use hybrid search
results_hybrid = ensemble_retriever.invoke(
    "What are the benefits of vector databases?"
)

In [17]:
def print_hybrid_results(results, query=None, preview_chars=300):
    """Gibt die Ergebnisse der Hybrid-Suche (Liste von Documents) formatiert aus."""
    print("=" * 80)
    print(f"Hybrid-Suche: {len(results)} Ergebnis(se)")
    if query:
        print(f"Anfrage: {query}")
    print("=" * 80)

    for rank, doc in enumerate(results, start=1):
        source = doc.metadata.get("source", "unbekannt")
        content = doc.page_content.replace("\n", " ").strip()
        if len(content) > preview_chars:
            content = content[:preview_chars] + "..."

        print(f"\n[{rank}] Quelle: {source}")
        print(f"    Länge: {len(doc.page_content)} Zeichen")
        print(f"    Inhalt: {content}")
        other_meta = {k: v for k, v in doc.metadata.items() if k != "source"}
        if other_meta:
            print(f"    Metadaten: {other_meta}")


print_hybrid_results(
    results_hybrid,
    query="What are the benefits of vector databases?",
)

Hybrid-Suche: 2 Ergebnis(se)
Anfrage: What are the benefits of vector databases?

[1] Quelle: c:\Artificial Intelligence\rag_tutorial\documents\ai-intro.txt
    Länge: 301 Zeichen
    Inhalt: predictive accuracy on held-out test data. The emergence of "Big Data" infrastructure—distributed storage systems and parallel processing frameworks—allowed algorithms to scale to web-scale corpora, fundamentally altering the economics of AI development from knowledge engineering to data acquisition...

[2] Quelle: c:\Artificial Intelligence\rag_tutorial\documents\neural-networks.txt
    Länge: 795 Zeichen
    Inhalt: # Neural Networks and Deep Learning  ## What is a Neural Network?  Artificial neural networks are computational models loosely inspired by the structure of biological brains. While the analogy to biology provides useful intuition, modern neural networks are best understood as mathematical function a...


### Semantic search

In [18]:
import re


def tokenize(text):
    # Wörter ohne Satzzeichen, kleingeschrieben
    return set(re.findall(r"\w+", text.lower()))


def calculate_custom_score(doc, query):
    # Simple keyword-overlap score between 0 and 1
    query_terms = tokenize(query)
    doc_terms = tokenize(doc.page_content)
    return len(query_terms & doc_terms) / len(query_terms) if query_terms else 0.0


In [19]:
def extract_highlights(doc, query):
    # Return sentences containing any query term
    query_terms = tokenize(query)
    return [
        sentence.strip()
        for sentence in doc.page_content.split(".")
        if query_terms & tokenize(sentence)
    ]


In [20]:
def advanced_search(query, filters=None, min_score=0.3, k=10):
    # Semantic search with optional metadata pre-filtering
    vector_results = vectorstore.similarity_search(
        query,
        k=k,
        filter=filters
    )

    # Post-process results
    processed_results = []
    for doc in vector_results:
        # Custom scoring logic
        relevance_score = calculate_custom_score(doc, query)
        if relevance_score >= min_score:
            processed_results.append({
                "document": doc,
                "score": relevance_score,
                "highlights": extract_highlights(doc, query)
            })

    return sorted(processed_results, key=lambda x: x["score"], reverse=True)


In [21]:
# Die Chunks haben nur das Metadatenfeld "source" -> Filter nur darauf (oder weglassen).
# Beispiel: filters={"source": {"$in": [str(documents_path / "ai-intro.txt")]}}
results_semantic = advanced_search(
    "machine learning applications",
    filters=None,
    min_score=0.3,
)


In [22]:
def print_semantic_results(results, preview_chars=300, max_highlights=3):
    """Gibt die Ergebnisse von advanced_search (Liste von Dicts) formatiert aus."""
    print("=" * 80)
    print(f"Erweiterte semantische Suche: {len(results)} Ergebnis(se)")
    print("=" * 80)

    if not results:
        print("Keine Treffer (Filter oder Score-Schwelle zu streng).")
        return

    for rank, item in enumerate(results, start=1):
        doc = item["document"]
        source = doc.metadata.get("source", "unbekannt")
        content = doc.page_content.replace("\n", " ").strip()
        if len(content) > preview_chars:
            content = content[:preview_chars] + "..."

        print(f"\n[{rank}] Score: {item['score']:.3f} | Quelle: {source}")
        print(f"    Inhalt: {content}")

        highlights = item["highlights"]
        if highlights:
            print(f"    Highlights ({len(highlights)}):")
            for sentence in highlights[:max_highlights]:
                print(f"      - {sentence}")
        other_meta = {k: v for k, v in doc.metadata.items() if k != "source"}
        if other_meta:
            print(f"    Metadaten: {other_meta}")


print_semantic_results(results_semantic)

Erweiterte semantische Suche: 10 Ergebnis(se)

[1] Score: 1.000 | Quelle: c:\Artificial Intelligence\rag_tutorial\documents\ai-intro.txt
    Inhalt: ## III. Core Technical Frameworks  The contemporary practice of machine learning organizes itself around distinct learning paradigms defined by the nature of supervision available during training. **Supervised learning** constitutes the predominant approach, wherein algorithms learn mappings from in...
    Highlights (3):
      - Core Technical Frameworks

The contemporary practice of machine learning organizes itself around distinct
learning paradigms defined by the nature of supervision available during
training
      - **Supervised learning** constitutes the predominant approach, wherein
algorithms learn mappings from input vectors $\mathbf{x} \in \mathcal{X}$ to
output labels $y \in \mathcal{Y}$ using datasets of the form $\mathcal{D} =
\{(\mathbf{x}_i, y_i)\}_{i=1}^n$
      - Classification
tasks involve discrete label spaces (predict